# deabook quickstart

A self-contained tour of the deabook API using synthetic data (no external files needed).

The pattern is always the same: **data → `sent` → factory → `.optimize()` → `DataFrame`**.

In [ ]:
import pandas as pd
from deabook import DEA, DEAweak, PANEL
from deabook.constant import RTS_VRS1, OPT_LOCAL, CONTEMPORARY

print("deabook ready")

In [ ]:
# Synthetic cross-section: 6 DMUs, 3 inputs (K, L, E) and 1 desirable output (Y)
data = pd.DataFrame({
    "K": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
    "L": [2.0, 3.0, 1.0, 4.0, 2.0, 5.0],
    "E": [3.0, 1.0, 2.0, 5.0, 4.0, 2.0],
    "Y": [5.0, 4.0, 3.0, 6.0, 7.0, 8.0],
})
data

In [ ]:
# 1) Standard radial efficiency (CCR, CRS, output-oriented)
m = DEA.ccr(data, sent="K L E=Y", orient="oo")
try:
    res = m.optimize(OPT_LOCAL, "highs")
    print(res[["optimization_status", "te"]].head())
except Exception as exc:  # e.g. solver not installed in this environment
    print("solve skipped:", type(exc).__name__, exc)
    print("model built successfully; install a solver (pip install highspy) to optimize.")

In [ ]:
# 2) SD model with an undesirable output (CO2) under weak disposability
data_bad = data.copy()
data_bad["CO2"] = [2.0, 1.0, 3.0, 4.0, 1.0, 2.0]

m = DEAweak.sddf(data_bad, sent="K L E=Y:CO2", rts=RTS_VRS1)
try:
    res = m.optimize(OPT_LOCAL, "highs")
    print(res[["optimization_status", "objective_value",
               "slack_x_K", "slack_y_Y", "slack_b_CO2"]].head())
except Exception as exc:
    print("solve skipped:", type(exc).__name__, exc)

In [ ]:
from deabook import PANEL
# 3) One observation per DMU and year.
panel_data = pd.concat([
    data.assign(country=[f"DMU{i}" for i in range(len(data))], year=year)
    for year in [2018, 2019]
], ignore_index=True)
p = PANEL.tech(
    panel_data, id="country", year="year", sent="K L E=Y",
    method=DEA.ccr, orient="oo", tech=CONTEMPORARY,
)
p.solve(solver="highs")
wide = p.pivot()
print(wide)
